In [1]:
# Permite importar src/ desde la carpeta notebooks/
import sys
from pathlib import Path
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

from sklearn.metrics import confusion_matrix
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.config import DIR_DATA_RAW

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)
sns.set_theme(style="whitegrid")

In [2]:
from src.config import DIR_DATA_PROC, DIR_MODELOS, RAIZ
from src.variables import construir_variables
import joblib

paquete = joblib.load(DIR_MODELOS / "actividad_v1.joblib")
sc = pd.read_parquet(RAIZ / "data" / "scoring" / "actividad_20260425_1400.parquet")

# Camino del notebook: la telemetría completa ya preparada en el paso 4
tel_v = pd.read_parquet(DIR_DATA_PROC / "telemetria_viajes.parquet")
nb = construir_variables(tel_v)
proba = paquete["modelo"].predict_proba(nb[paquete["variables"]])
nb["Actividad_nb"] = paquete["modelo"].classes_[proba.argmax(axis=1)]
nb["Prob_nb"] = proba.max(axis=1)

comp = sc.merge(nb[["ep", "fecha", "Actividad_nb", "Prob_nb"]], on=["ep", "fecha"], how="left")
print(f"Registros del scoring: {len(sc)} · encontrados en el notebook: {comp['Actividad_nb'].notna().sum()}")
print(f"Misma actividad: {(comp['Actividad'] == comp['Actividad_nb']).mean():.1%}")
print(f"Diferencia máxima de probabilidad: {(comp['Probabilidad'] - comp['Prob_nb']).abs().max():.4f}")
display(comp[comp["Actividad"] != comp["Actividad_nb"]][["ep", "fecha", "Velocidad", "Actividad", "Actividad_nb", "Probabilidad", "Prob_nb"]])

Registros del scoring: 121 · encontrados en el notebook: 121
Misma actividad: 100.0%
Diferencia máxima de probabilidad: 0.0000


,ep,fecha,Velocidad,Actividad,Actividad_nb,Probabilidad,Prob_nb
